# Testing code for how to read a patch from a raster file rather than the whole file

In [12]:
from typing import Final
from pathlib import Path
from os import getenv
from dotenv import find_dotenv, load_dotenv
from rasterio import open as open_raster
from rasterio.windows import Window
from PIL import Image

PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))
IMG_DIR = LOCAL_DIR\
    .joinpath("outputs/manual-labelling/processed/glam-st17ne-2.tif")
DEST_DIR = LOCAL_DIR\
    .joinpath("outputs/manual-labelling/merge-trial/glam-st17ne-2.tif")

# Windowing
with open_raster(IMG_DIR) as src:

    # The size in pixels of the window
    xsize, ysize = 6000, 4500

    # Create a Window and calculate the transform from the source dataset    
    window = Window(0, 0, xsize, ysize)
    transform = src.window_transform(window)

    # Create a new cropped raster to write to
    profile = src.profile
    profile.update({
        'height': ysize,
        'width': xsize,
        'transform': transform
    })

    # Save out windowed raster
    with open_raster(DEST_DIR, 'w', **profile) as dst:
        # Read the data from the window and write it to the output raster
        dst.write(src.read(window = window))

In [ ]:
with open_raster(DEST_DIR) as src:
    data = (-src.read() + 1) * 255 # image array

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize = (15, 15))
ax.imshow(data[0], cmap = "grey")

In [14]:
Image\
    .fromarray(data[0], mode = "L")\
    .save(DEST_DIR.parent.joinpath(f"{DEST_DIR.stem}.png"))